# Chapter 2: Attention Mechanisms and Positional Encoding
**Module 04: Introduction to LLMs in Python**

> Source integrated from `chapter2.pdf`.

## Learning Objectives
- Explain why attention solves long-range dependency issues.
- Implement sinusoidal positional encoding.
- Build multi-head self-attention with Q, K, and V projections.
- Assemble encoder, decoder, and encoder-decoder transformer blocks.

## Chapter Map
| Section | Focus |
|---|---|
| 2.1 | Attention intuition |
| 2.2 | Positional encoding |
| 2.3 | Multi-head self-attention |
| 2.4 | Encoder-only transformer |
| 2.5 | Decoder-only transformer and causal masks |
| 2.6 | Encoder-decoder transformer and cross-attention |


## 2.1 Why Attention Mechanisms?

Sequential models struggle to retain information from far back in long sequences. Attention lets every token compare itself with other tokens and decide which positions matter.

| Projection | Question It Answers |
|---|---|
| Query (Q) | What am I looking for? |
| Key (K) | What information do I contain for matching? |
| Value (V) | What information should be passed forward? |

> **Tip:** Attention behaves like differentiable retrieval: query-key similarity decides how much of each value is used.


## 2.2 Positional Encoding

Transformers process tokens in parallel, so they need explicit position information. A positional encoding vector `PE` is added to each token embedding `E`.

| Dimension | Formula Family |
|---|---|
| Even indices | Sine |
| Odd indices | Cosine |

The positional encoding matrix is registered as a non-trainable buffer.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length=512):
        super().__init__()
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]

# Alias matching the PDF slide name.
PositionalEncoder = PositionalEncoding

x = torch.zeros(2, 10, 16)
print(PositionalEncoding(16, 50)(x).shape)


## 2.3 Multi-Head Self-Attention

Multi-head attention splits embeddings across several heads. Each head learns a different attention pattern, then the outputs are concatenated and projected.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads
        self.query_linear = nn.Linear(d_model, d_model)
        self.key_linear = nn.Linear(d_model, d_model)
        self.value_linear = nn.Linear(d_model, d_model)
        self.output_linear = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.head_dim)
        return x.permute(0, 2, 1, 3).contiguous().view(batch_size * self.num_heads, -1, self.head_dim)

    def compute_attention(self, query, key, mask=None):
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            if mask.dim() == 3:
                if mask.size(0) == 1:
                    mask = mask.repeat(query.size(0), 1, 1)
                elif mask.size(0) * self.num_heads == query.size(0):
                    mask = mask.repeat_interleave(self.num_heads, dim=0)
            scores = scores.masked_fill(mask == 0, float("-1e9"))
        return F.softmax(scores, dim=-1)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        query = self.split_heads(self.query_linear(query), batch_size)
        key = self.split_heads(self.key_linear(key), batch_size)
        value = self.split_heads(self.value_linear(value), batch_size)
        attention_weights = self.compute_attention(query, key, mask)
        output = torch.matmul(attention_weights, value)
        output = output.view(batch_size, self.num_heads, -1, self.head_dim)
        output = output.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.d_model)
        return self.output_linear(output)

attention = MultiHeadAttention(d_model=16, num_heads=4)
sample = torch.randn(2, 5, 16)
print(attention(sample, sample, sample).shape)


## 2.4 Building an Encoder Transformer

An encoder stack contains embeddings, positional encoding, repeated encoder layers, and a task head. Encoder layers use self-attention, feed-forward layers, residual connections, layer normalization, and dropout.


In [ ]:
class FeedForwardSubLayer(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForwardSubLayer(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        return self.norm2(x + self.dropout(ff_output))

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_sequence_length):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_sequence_length)
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

    def forward(self, x, mask=None):
        x = self.positional_encoding(self.embedding(x))
        for layer in self.layers:
            x = layer(x, mask)
        return x

encoder = TransformerEncoder(1000, 32, 2, 4, 64, 0.1, 128)
tokens = torch.randint(0, 1000, (2, 12))
encoded = encoder(tokens)
print(encoded.shape)


In [ ]:
class ClassifierHead(nn.Module):
    def __init__(self, d_model, num_classes):
        super().__init__()
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        logits = self.fc(x)
        return F.log_softmax(logits, dim=-1)

class RegressionHead(nn.Module):
    def __init__(self, d_model, output_dim):
        super().__init__()
        self.fc = nn.Linear(d_model, output_dim)

    def forward(self, x):
        return self.fc(x)

print(ClassifierHead(32, 3)(encoded[:, 0, :]).shape)
print(RegressionHead(32, 1)(encoded[:, 0, :]).shape)


## 2.5 Decoder-Only Transformers and Causal Masks

Decoder-only models generate text autoregressively. A triangular mask hides future positions so a token can attend only to itself and earlier tokens.


In [ ]:
sequence_length = 6
self_attention_mask = (1 - torch.triu(torch.ones(1, sequence_length, sequence_length), diagonal=1)).bool()
print(self_attention_mask[0].int())


In [ ]:
class DecoderOnlyLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForwardSubLayer(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, self_mask=None):
        attn_output = self.self_attn(x, x, x, self_mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        return self.norm2(x + self.dropout(ff_output))

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_sequence_length):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_sequence_length)
        self.layers = nn.ModuleList([DecoderOnlyLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x, self_mask=None):
        x = self.positional_encoding(self.embedding(x))
        for layer in self.layers:
            x = layer(x, self_mask)
        return F.log_softmax(self.fc(x), dim=-1)

decoder_only = DecoderOnlyTransformer(1000, 32, 2, 4, 64, 0.1, 128)
mask = (1 - torch.triu(torch.ones(1, tokens.size(1), tokens.size(1)), diagonal=1)).bool()
print(decoder_only(tokens, mask).shape)


## 2.6 Encoder-Decoder Transformers and Cross-Attention

Cross-attention uses decoder states as queries and encoder outputs as keys and values. This lets the decoder look back at the processed input sequence while generating output tokens.


In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForwardSubLayer(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, y, causal_mask=None, cross_mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, causal_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, y, y, cross_mask)))
        return self.norm3(x + self.dropout(self.feed_forward(x)))

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_sequence_length):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_sequence_length)
        self.layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x, encoder_output, causal_mask=None, cross_mask=None):
        x = self.positional_encoding(self.embedding(x))
        for layer in self.layers:
            x = layer(x, encoder_output, causal_mask, cross_mask)
        return F.log_softmax(self.fc(x), dim=-1)

class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_len, dropout):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_len)
        self.decoder = TransformerDecoder(vocab_size, d_model, num_layers, num_heads, d_ff, dropout, max_seq_len)

    def forward(self, src, tgt, src_mask=None, causal_mask=None, cross_mask=None):
        encoder_output = self.encoder(src, src_mask)
        return self.decoder(tgt, encoder_output, causal_mask, cross_mask)

seq2seq = Transformer(1000, 32, 4, 2, 64, 128, 0.1)
source = torch.randint(0, 1000, (2, 8))
target = torch.randint(0, 1000, (2, 6))
causal_mask = (1 - torch.triu(torch.ones(1, target.size(1), target.size(1)), diagonal=1)).bool()
print(seq2seq(source, target, causal_mask=causal_mask).shape)


## Chapter Summary
- Positional encodings inject order into parallel token processing.
- Self-attention uses Q, K, and V projections to produce contextual token representations.
- Multi-head attention learns several relationship patterns at once.
- Encoder-only models suit classification and extractive QA.
- Decoder-only models use causal masks for generation.
- Encoder-decoder models use cross-attention for translation and summarization.
